# 컴퓨터비전 부트캠프 · 1일차 실습
## CIFAR-10 영상 분류 — baseline 모델 만들기

부경대학교 교내 컴퓨터비전 부트캠프 · 2026. 8. 3.
김한울 (서울과학기술대학교 인공지능응용학과)

---

### 이 실습에서 하는 일

32×32 크기의 컬러 이미지를 10개 클래스로 분류하는 CNN을 **처음부터 끝까지 직접** 만들어 봅니다.
오전 이론 시간에 본 파이프라인을 그대로 코드로 옮기는 과정입니다.

| 단계 | 내용 | 예상 시간 |
|---|---|---|
| STEP 0 | 환경 확인 (GPU 연결) | 2분 |
| STEP 1 | 데이터 준비 — transform, DataLoader | 8분 |
| STEP 2 | 데이터 살펴보기 — 이미지·분포 확인 | 5분 |
| STEP 3 | 모델 정의 — SimpleCNN | 10분 |
| STEP 4 | 학습 — 학습 루프 작성과 실행 | 12분 |
| STEP 5 | 평가 — 정확도·학습 곡선·혼동 행렬 | 8분 |
| STEP 6 | 오류 분석 — 틀린 이미지 확인 | 5분 |

### 오늘의 목표

- 검증 정확도(validation accuracy) **65% 이상** 달성하기
- 어떤 클래스에서 왜 틀리는지 **말로 설명할 수 있게** 되기
- 내일 해커톤에서 시도할 개선 아이디어를 **2개 이상** 적어 오기

### 진행 방법

- 셀은 반드시 **위에서부터 순서대로** 실행합니다 (`Shift + Enter`).
- `# TODO` 가 붙은 줄은 직접 작성합니다. 바로 위에 힌트가 있습니다.
- 5분 이상 막히면 손을 들어 주세요.

> **먼저 할 일 — 런타임을 GPU로 바꾸기**
> 상단 메뉴 `런타임` → `런타임 유형 변경` → 하드웨어 가속기 `T4 GPU` → 저장

---
## STEP 0. 환경 확인

아래 셀을 실행했을 때 `cuda` 가 출력되어야 합니다.
`cpu` 가 나오면 런타임 유형이 GPU로 설정되지 않은 것입니다.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch 버전 :", torch.__version__)
print("사용할 장치   :", device)
if device.type == "cuda":
    print("GPU 이름      :", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU가 아닙니다. 런타임 → 런타임 유형 변경 → T4 GPU 로 바꿔 주세요.")

### 라이브러리 불러오기와 시드 고정

시드를 고정하면 실행할 때마다 (거의) 같은 결과가 나와 실험 비교가 쉬워집니다.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

plt.rcParams["figure.dpi"] = 110
print("준비 완료")

---
## STEP 1. 데이터 준비

### 1-1. transform 정의

이미지를 모델에 넣기 전 두 가지를 반드시 거칩니다.

| 변환 | 하는 일 |
|---|---|
| `transforms.ToTensor()` | `(H, W, 3)` 0~255 정수 → `(3, H, W)` 0.0~1.0 실수 |
| `transforms.Normalize(mean, std)` | 채널별로 평균 0 · 표준편차 1 에 가깝게 만든다 |

아래 `CIFAR10_MEAN`, `CIFAR10_STD` 는 CIFAR-10 학습 데이터 전체에서 미리 계산해 둔 값입니다.

> 오늘은 **데이터 증강을 쓰지 않습니다.** baseline 성능을 먼저 확인하고,
> 내일 증강을 넣었을 때 얼마나 좋아지는지 비교하기 위해서입니다.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

# TODO: ToTensor() 와 Normalize() 를 순서대로 담은 Compose 를 만드세요.
#       힌트: transforms.Compose([ transforms.ToTensor(), transforms.Normalize(평균, 표준편차) ])
transform = transforms.Compose([
    # 여기를 채우세요
])

print(transform)

### 1-2. CIFAR-10 내려받기

처음 실행하면 약 170MB를 내려받습니다 (1~2분).

In [ ]:
full_train = datasets.CIFAR10(root="./data", train=True,
                              download=True, transform=transform)
test_set   = datasets.CIFAR10(root="./data", train=False,
                              download=True, transform=transform)

CLASSES = full_train.classes
print("클래스 :", CLASSES)
print("train 전체 :", len(full_train), "장")
print("test       :", len(test_set), "장")

### 1-3. train / validation 분리

CIFAR-10은 train 50,000장과 test 10,000장으로 이미 나뉘어 있습니다.
하지만 **test는 마지막에 단 한 번만** 써야 하므로,
train에서 5,000장을 떼어 validation으로 사용합니다.

In [ ]:
train_set, val_set = random_split(
    full_train, [45000, 5000],
    generator=torch.Generator().manual_seed(SEED)
)

print("train      :", len(train_set), "장")
print("validation :", len(val_set), "장")
print("test       :", len(test_set), "장")

### 1-4. DataLoader 만들기

`DataLoader`는 Dataset에서 이미지를 꺼내 **배치 단위로 묶어** 줍니다.

| 인자 | 의미 | 권장값 |
|---|---|---|
| `batch_size` | 한 번에 처리할 이미지 수 | 128 (메모리 부족 시 64) |
| `shuffle` | 매 epoch 순서를 섞을지 | **train만 True** |
| `num_workers` | 데이터를 읽는 병렬 프로세스 수 | Colab에서는 2 |

> `shuffle=True`를 빠뜨리면 배치마다 비슷한 데이터만 들어와 학습이 불안정해집니다.

In [ ]:
BATCH_SIZE = 128

# TODO: 세 개의 DataLoader 를 만드세요.
#       train_loader 만 shuffle=True 이고, 나머지는 shuffle=False 입니다.
#       힌트: DataLoader(데이터셋, batch_size=..., shuffle=..., num_workers=2, pin_memory=True)
train_loader = None   # 여기를 채우세요 (batch_size=BATCH_SIZE, shuffle=True)
val_loader   = None   # 여기를 채우세요 (batch_size=256,        shuffle=False)
test_loader  = None   # 여기를 채우세요 (batch_size=256,        shuffle=False)

print("배치 수 — train:", len(train_loader),
      "/ val:", len(val_loader), "/ test:", len(test_loader))

---
## STEP 2. 데이터 살펴보기

**모델을 만들기 전에 데이터를 눈으로 보는 습관**이 중요합니다.
데이터가 이상하면 모델을 아무리 고쳐도 성능이 오르지 않습니다.

### 2-1. 배치의 모양 확인

In [ ]:
images, labels = next(iter(train_loader))

print("images.shape :", images.shape)   # (배치, 채널, 높이, 너비)
print("labels.shape :", labels.shape)
print("픽셀 값 범위 : %.2f ~ %.2f  (정규화 후라 음수가 나옵니다)"
      % (images.min().item(), images.max().item()))
print("첫 이미지의 정답 :", labels[0].item(), "=", CLASSES[labels[0]])

### 2-2. 샘플 이미지 그려 보기

정규화된 텐서를 그대로 그리면 색이 이상하게 보이므로, 되돌린 뒤(`unnormalize`) 그립니다.

In [ ]:
def unnormalize(img_tensor):
    """정규화된 (3,H,W) 텐서를 0~1 범위의 (H,W,3) 배열로 되돌린다."""
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std  = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    img = img_tensor.cpu() * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


fig, axes = plt.subplots(4, 8, figsize=(12, 6.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(unnormalize(images[i]))
    ax.set_title(CLASSES[labels[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10 sample images", fontsize=14)
plt.tight_layout()
plt.show()

> **잠깐 생각해 보기** — 32×32는 생각보다 작습니다.
> 사람인 여러분도 헷갈리는 이미지가 보이나요? 어떤 클래스끼리 헷갈리나요?

### 2-3. 클래스 분포 확인

한 클래스만 지나치게 많으면 정확도가 왜곡됩니다. CIFAR-10은 균형이 맞는지 확인합니다.

In [ ]:
train_labels = np.array([full_train.targets[i] for i in train_set.indices])
counts = np.bincount(train_labels, minlength=10)

plt.figure(figsize=(9, 3.2))
plt.bar(CLASSES, counts, color="#156082")
plt.xticks(rotation=45, ha="right")
plt.ylabel("count")
plt.title("Class distribution (train split)")
plt.tight_layout()
plt.show()

print("클래스별 장수 :", counts)
print("최대/최소 비율 : %.2f  (1.0 에 가까울수록 균형)" % (counts.max() / counts.min()))

---
## STEP 3. 모델 정의

이론 시간에 본 구조를 그대로 만듭니다.

```
입력 3×32×32
  → Conv(3→32) + ReLU + MaxPool   →  32×16×16
  → Conv(32→64) + ReLU + MaxPool  →  64×8×8
  → Conv(64→128) + ReLU + MaxPool → 128×4×4
  → Flatten                        → 2048
  → Linear(2048→256) + ReLU + Dropout
  → Linear(256→10)                 → 클래스 점수 10개
```

`nn.Module`을 상속할 때 채워야 할 것은 두 개뿐입니다.

- `__init__` : **어떤 부품을 쓸지**
- `forward` : **그 부품을 어떤 순서로 통과시킬지**

> 마지막에 `softmax`를 붙이지 않습니다. `nn.CrossEntropyLoss`가 내부에 이미 포함하고 있습니다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: Conv2d -> ReLU -> MaxPool2d 를 3번 반복하는 Sequential 을 만드세요.
        #       채널 수는 3 -> 32 -> 64 -> 128, 모두 kernel_size=3, padding=1 입니다.
        #       MaxPool2d(2) 를 한 번 통과할 때마다 가로·세로가 절반이 됩니다.
        self.features = nn.Sequential(
            # 여기를 채우세요
        )

        # TODO: Flatten -> Linear(2048, 256) -> ReLU -> Dropout(0.3) -> Linear(256, num_classes)
        #       힌트: 128 x 4 x 4 = 2048
        self.classifier = nn.Sequential(
            # 여기를 채우세요
        )

    def forward(self, x):
        # TODO: features 를 통과시킨 뒤 classifier 를 통과시켜 반환하세요.
        pass   # 여기를 채우세요


model = SimpleCNN().to(device)
print(model)

### 3-1. 모델이 제대로 만들어졌는지 확인

가짜 입력을 한 번 통과시켜 **출력 모양이 (배치, 10)** 인지 확인합니다.
이 확인을 습관화하면 학습을 돌리기 전에 대부분의 구조 오류를 잡을 수 있습니다.

In [ ]:
dummy = torch.randn(4, 3, 32, 32).to(device)   # 가짜 이미지 4장
out = model(dummy)

print("입력 :", tuple(dummy.shape))
print("출력 :", tuple(out.shape), "  <- (배치 4, 클래스 10) 이어야 합니다")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("학습 가능한 파라미터 수 : {:,} 개".format(n_params))

---
## STEP 4. 학습

### 4-1. 손실함수와 옵티마이저

| 구성 | 선택 | 이유 |
|---|---|---|
| 손실함수 | `nn.CrossEntropyLoss()` | 다중 클래스 분류의 표준. softmax를 내부에 포함 |
| 옵티마이저 | `optim.AdamW(lr=1e-3)` | 파라미터마다 보폭을 자동 조절해 손이 덜 간다 |

In [ ]:
# TODO: 손실함수와 옵티마이저를 만드세요.
#       힌트: nn.CrossEntropyLoss() / optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = None   # 여기를 채우세요
optimizer = None   # 여기를 채우세요

print(criterion)
print(optimizer)

### 4-2. 한 epoch 학습하는 함수

이론 시간에 본 **다섯 줄**이 그대로 들어갑니다.

```
optimizer.zero_grad()   ① 이전 기울기 초기화
outputs = model(images) ② 순전파
loss = criterion(...)   ③ 손실 계산
loss.backward()         ④ 역전파
optimizer.step()        ⑤ 파라미터 갱신
```

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()                      # 학습 모드 (Dropout 켜짐)
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # TODO: 학습 5단계를 순서대로 작성하세요.
        #   ① optimizer.zero_grad()
        #   ② outputs = model(images)
        #   ③ loss = criterion(outputs, labels)
        #   ④ loss.backward()
        #   ⑤ optimizer.step()
        # 여기를 채우세요

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total

### 4-3. 평가하는 함수

검증·테스트에서는 두 가지를 반드시 켭니다.

- `model.eval()` — Dropout을 끄고 BatchNorm을 고정 통계로 전환
- `torch.no_grad()` — 기울기 그래프를 만들지 않아 빠르고 메모리를 아낀다

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    # TODO: 평가 모드로 전환하세요.  힌트: model.eval()
    # 여기를 채우세요

    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)
        # TODO: 맞힌 개수를 correct 에 더하세요.
        #       힌트: outputs.argmax(dim=1) 이 예측 클래스입니다.
        correct += 0   # 여기를 고치세요
        total   += labels.size(0)

    return running_loss / total, correct / total

### 4-4. 학습 실행

GPU에서 10 epoch에 약 2분 걸립니다. (CPU라면 20분 이상 — 반드시 GPU로 바꾸세요)

매 epoch마다 train / validation 손실과 정확도를 기록해 두었다가 나중에 그래프로 그립니다.

In [ ]:
EPOCHS = 10
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    if va_acc > best_val_acc:                      # 가장 좋았던 시점의 모델을 보관
        best_val_acc = va_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        mark = "  ← best"
    else:
        mark = ""

    print(f"epoch {epoch:2d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f}{mark}")

print(f"\n최고 검증 정확도 : {best_val_acc:.4f}")

---
## STEP 5. 평가

### 5-1. 학습 곡선

**train loss만 보면 과적합을 발견할 수 없습니다.** 반드시 validation과 함께 봅니다.

- train↓ / val↓ → 잘 학습되는 중
- train↓ / val↑ → **과적합** — 이 지점부터는 학습을 멈추는 게 낫다

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], "-o", label="train", color="#156082")
axes[0].plot(epochs, history["val_loss"],   "-o", label="validation", color="#E97132")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[0].grid(color="#EEEEEE")

axes[1].plot(epochs, history["train_acc"], "-o", label="train", color="#156082")
axes[1].plot(epochs, history["val_acc"],   "-o", label="validation", color="#E97132")
axes[1].axhline(0.65, ls="--", color="#999999")
axes[1].text(1, 0.66, "target 0.65", fontsize=9, color="#666666")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
axes[1].grid(color="#EEEEEE")

plt.tight_layout()
plt.show()

### 5-2. 테스트 정확도

가장 좋았던 시점의 모델을 되돌린 뒤, **test set으로 단 한 번** 측정합니다.

In [ ]:
model.load_state_dict(best_state)
model.to(device)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"test loss {test_loss:.4f} | test accuracy {test_acc:.4f}")

if test_acc >= 0.65:
    print("목표(0.65) 달성!")
else:
    print("목표에 조금 못 미칩니다. epoch 를 늘리거나 learning rate 를 조정해 보세요.")

### 5-3. 예측 모으기

혼동 행렬과 오류 분석에 쓸 예측 결과를 한 번에 모아 둡니다.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels, all_images = [], [], []
    for images, labels in loader:
        outputs = model(images.to(device))
        all_preds.append(outputs.argmax(dim=1).cpu())
        all_labels.append(labels)
        all_images.append(images)
    return (torch.cat(all_preds).numpy(),
            torch.cat(all_labels).numpy(),
            torch.cat(all_images))


preds, targets, test_images = collect_predictions(model, test_loader, device)
print("예측 :", preds.shape, "/ 정답 :", targets.shape)

### 5-4. 혼동 행렬

**행 = 정답, 열 = 예측**입니다. 대각선이 진할수록 좋고,
대각선 밖의 진한 칸이 모델이 반복적으로 헷갈리는 쌍입니다.

In [ ]:
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(targets, preds):
    cm[t, p] += 1

cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (normalized by row)")

for i in range(10):
    for j in range(10):
        if cm_norm[i, j] >= 0.03:
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color="white" if cm_norm[i, j] > 0.5 else "#333333")

fig.colorbar(im, ax=ax, fraction=0.045)
plt.tight_layout()
plt.show()

### 5-5. 클래스별 정확도와 가장 헷갈린 쌍

In [ ]:
per_class = cm.diagonal() / cm.sum(axis=1)
order = np.argsort(per_class)

print("클래스별 정확도 (낮은 순)")
for i in order:
    bar = "█" * int(per_class[i] * 40)
    print(f"  {CLASSES[i]:<12} {per_class[i]:.3f}  {bar}")

print("\n가장 많이 헷갈린 쌍 TOP 5")
confusions = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j]
for n, i, j in sorted(confusions, reverse=True)[:5]:
    print(f"  실제 {CLASSES[i]:<12} → {CLASSES[j]:<12} {n:4d} 장")

---
## STEP 6. 오류 분석

숫자만 보지 말고 **틀린 이미지를 직접 눈으로** 봅니다.
"모델이 왜 틀렸는지"를 설명할 수 있어야 다음 개선 방향이 나옵니다.

In [ ]:
wrong_idx = np.where(preds != targets)[0]
print(f"틀린 이미지 : {len(wrong_idx)} / {len(targets)} 장 "
      f"({len(wrong_idx) / len(targets):.1%})")

rng = np.random.default_rng(0)
sample = rng.choice(wrong_idx, size=16, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(15, 4.6))
for ax, idx in zip(axes.flat, sample):
    ax.imshow(unnormalize(test_images[idx]))
    ax.set_title(f"T: {CLASSES[targets[idx]]}\nP: {CLASSES[preds[idx]]}",
                 fontsize=9, color="#B03A2E")
    ax.axis("off")
plt.suptitle("Misclassified samples  (T = true, P = predicted)", fontsize=13)
plt.tight_layout()
plt.show()

### 직접 답해 보기

아래 셀(마크다운)을 더블클릭해서 직접 적어 보세요. 내일 해커톤의 출발점이 됩니다.

1. 가장 정확도가 낮은 클래스는 무엇이고, 어떤 클래스와 헷갈렸나요?

    →

2. 틀린 이미지들을 보고 짐작한 원인은 무엇인가요? (해상도 / 배경 / 자세 / 색 등)

    →

3. 성능을 올리기 위해 내일 시도해 보고 싶은 아이디어 2가지는?

    →
    →

---
## 도전 과제 (시간이 남으면)

아래는 **내일 해커톤에서 본격적으로 다룰 내용**의 맛보기입니다.
지금 시도해 보고 baseline과 얼마나 차이가 나는지 기록해 두세요.

| # | 과제 | 힌트 |
|---|---|---|
| 1 | epoch를 20으로 늘려 보기 | 과적합이 언제부터 시작되는지 학습 곡선으로 확인 |
| 2 | learning rate를 1e-2 / 1e-4로 바꿔 보기 | 너무 크면 발산, 너무 작으면 느림 |
| 3 | 데이터 증강 추가하기 | `transforms.RandomHorizontalFlip()`, `RandomCrop(32, padding=4)` |
| 4 | 채널 수를 64 → 128 → 256으로 늘려 보기 | 파라미터 수와 학습 시간이 얼마나 늘어나는지도 함께 기록 |
| 5 | `weight_decay`를 0 / 1e-2 / 5e-2로 바꿔 보기 | 값이 클수록 과적합은 줄지만 학습이 덜 된다 |

> **기록 요령** — 한 번에 하나씩만 바꾸고 결과를 표로 적으세요.
> 두 개를 동시에 바꾸면 무엇이 효과가 있었는지 알 수 없습니다.

| 시도 | 바꾼 것 | val accuracy | 메모 |
|---|---|---|---|
| baseline | — | | |
| 1 | | | |
| 2 | | | |

---
## 마무리 체크리스트

- [ ] 검증 정확도 65% 이상을 달성했다
- [ ] 학습 곡선을 보고 과적합 여부를 판단할 수 있다
- [ ] 혼동 행렬에서 가장 헷갈리는 클래스 쌍을 찾았다
- [ ] 틀린 이미지를 직접 보고 원인을 한 문장으로 설명했다
- [ ] 내일 시도할 개선 아이디어를 2개 이상 적었다

수고하셨습니다. 내일은 **성능 개선과 전이학습, 그리고 영상분류 해커톤**입니다.